# A Deno-style notebook

Written against `deno jupyter`'s APIs. Executed by cdp-kernel on a Deno target and on a Node target with the Deno compatibility shim.

In [ ]:
// The compat shim reports a placeholder Deno version.
const runtime = Deno.version.deno.includes("cdp-kernel")
  ? `Node ${process.versions.node} (Deno compat shim)` : `Deno ${Deno.version.deno}`;
Deno.jupyter.md`Running on **${runtime}**, V8 ${Deno.version.v8}`

In [ ]:
import pl from "npm:nodejs-polars@0.18.0";

const df = pl.DataFrame({
  fruit: ["apple", "banana", "cherry", "durian"],
  count: [3, 5, 8, 1],
  price: [0.5, 0.25, 3.0, 9.75],
});
df

In [ ]:
df.filter(pl.col("count").gt(2)).withColumn(pl.col("price").mul(pl.col("count")).alias("total"))

In [ ]:
interface Point { x: number; y: number }
const points: Point[] = Array.from({ length: 12 }, (_, i) => ({ x: i, y: Math.round(10 * Math.sin(i / 2)) }));
const chart = {
  toSpec: () => ({
    $schema: "https://vega.github.io/schema/vega-lite/v5.json",
    data: { values: points },
    mark: "line",
    encoding: { x: { field: "x", type: "quantitative" }, y: { field: "y", type: "quantitative" } },
  }),
};
chart

In [ ]:
Deno.jupyter.svg`<svg width="120" height="40" xmlns="http://www.w3.org/2000/svg">
  <rect width="120" height="40" rx="6" fill="#2f5d50"/>
  <text x="60" y="26" fill="#f4efe3" font-family="sans-serif" font-size="16" text-anchor="middle">cdp-kernel</text>
</svg>`

In [ ]:
await Deno.jupyter.display(Deno.jupyter.html`<progress value="0" max="3"></progress>`, { display_id: "progress" });
for (let i = 1; i <= 3; i++) {
  await new Promise((r) => setTimeout(r, 50));
  await Deno.jupyter.display(Deno.jupyter.html`<progress value="${i}" max="3"></progress> step ${i}`, { display_id: "progress", update: true });
}
console.log("done");

In [ ]:
class Badge {
  constructor(readonly label: string) {}
  [Symbol.for("Jupyter.display")]() {
    return { "text/html": `<span style="padding:2px 8px;border-radius:9px;background:#b08d57;color:#fff">${this.label}</span>` };
  }
}
new Badge("custom display")

In [ ]:
const dir = await Deno.makeTempDir({ prefix: "nb-" });
await Deno.writeTextFile(`${dir}/hello.txt`, "hello from a notebook\n");
const names = [];
for await (const entry of Deno.readDir(dir)) names.push(entry.name);
({ names, text: await Deno.readTextFile(`${dir}/hello.txt`) })